In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math

# ==============================================================================
# 0. SymPy settings
# ==============================================================================
sp.init_printing(use_unicode=True)
z = sp.Symbol('z', complex=True)
q = sp.Symbol('q', complex=True)
n = sp.Symbol('n', integer=True)

print("=== LTI System Zero-Input Response Analysis — Problem 090604 ===")
print()

# ==============================================================================
# 1. UNILATERAL Z-TRANSFORM Y^+(z) & SYMBOLIC PFE
# ==============================================================================
# Difference equation: y[n] - 1.5*y[n-1] + 0.5*y[n-2] = 0
# Initial conditions: y[-1] = 1, y[-2] = 0
# Unilateral transform expression in terms of q = z^(-1)
Y_z_plus_q = ( sp.Rational(3, 2) - sp.Rational(1, 2) * q ) / (1 - sp.Rational(3, 2) * q + sp.Rational(1, 2) * q**2)

display(sp.Eq(sp.Symbol("Y^+(z)"), Y_z_plus_q.subs(q, z**(-1))))

# Find roots of denominator in terms of q = z^(-1)
den_q = 1 - sp.Rational(3, 2)*q + sp.Rational(1, 2)*q**2
q_roots = sp.solve(den_q, q)

# Poles in z: p1 = 1/q1, p2 = 1/q2
p1 = sp.simplify(1 / q_roots[0])
p2 = sp.simplify(1 / q_roots[1])

# Trial PFE form with symbolic coefficients A1, A2
A1, A2 = sp.symbols('A1 A2')
Y_trial_q = A1 / (1 - p1 * q) + A2 / (1 - p2 * q)

# Match numerators to find A1, A2 robustly
num_q = sp.expand(sp.together(Y_trial_q - Y_z_plus_q).as_numer_denom()[0])
poly_q = sp.Poly(num_q, q)
equations = [sp.Eq(coeff, 0) for coeff in poly_q.all_coeffs()]
solution = sp.solve(equations, [A1, A2], dict=True)[0]

A1_val = sp.simplify(solution[A1])
A2_val = sp.simplify(solution[A2])

# Fully symbolic PFE form
Y_pfe_z = (A1 / (1 - p1 * z**(-1)) + A2 / (1 - p2 * z**(-1))).subs(solution)

display(Math(r"Y^+(z) \text{ in PFE form (fully symbolic):}"))
display(Y_pfe_z)

# Fully symbolic analytical zero-input response y[n]
y_zi = sp.simplify((A1_val * p1**n + A2_val * p2**n))

display(Math(r"y[n] \text{ (symbolic expression from SymPy):}"))
display(y_zi)

print("\nEquivalent analytical form:")
display(Math(r"y[n] = \left[2 - (0.5)^{n+1}\right]u[n]"))

# ==============================================================================
# 2. NUMERICAL EVALUATION & PLOTTING
# ==============================================================================
def evaluate_symbolic(expr, n_values):
    return np.array([float(sp.re(sp.N(expr.subs(n, int(k))))) for k in n_values])

out = widgets.Output()

def plot_system_response(N=20):
    with out:
        out.clear_output(wait=True)
        n_vec = np.arange(0, N)
        zi_vals = evaluate_symbolic(y_zi, n_vec)

        fig, axes = plt.subplots(1, 1, figsize=(10, 4))

        axes.stem(n_vec, zi_vals, basefmt=" ")
        axes.set_title(r"Zero-Input Response $y[n]$ with Initial Conditions", fontsize=11, fontweight='bold')
        axes.set_xlabel(r"$n$")
        axes.set_ylabel(r"$y[n]$")
        axes.grid(True)

        plt.tight_layout()
        plt.show()

plot_system_response()
display(out)